# Notebook 1 — Chicago Roastery: Floor Structure & Menu Data

**Goal:** Digitize the Starbucks Reserve Roastery Chicago into a structured dataset.

The Chicago Roastery is the world's largest Starbucks — 35,000 sq ft across 5 floors at 646 N Michigan Ave. Unlike a regular Starbucks optimized for throughput, this is a **coffee theme park** designed for exploration and brand immersion.

This notebook creates the foundational data layer for the Agent-Based Model:
1. Floor structure with features and functions
2. Menu items per floor with nutrition data
3. Estimated price ranges from public sources
4. Adjacency matrix for floor connectivity

**Data Sources:** [Starbucks Reserve official site](https://www.starbucksreserve.com/locations/chicago-roastery), [TravelFamilyBlog floor guide](https://travelfamilyblog.com/starbucks-reserve-roastery-chicago-a-complete-guide-to-every-floor/), media coverage, Wikipedia

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from IPython.display import display

np.random.seed(42)

OUT_DIR = Path('../data/raw')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## Section 1 — Floor Structure

| Floor | Name | Function | Key Feature | Atmosphere |
|-------|------|----------|-------------|------------|
| 1F | Main Reserve Bar & Scooping Bar | Coffee, pastries, retail | Roasting operations visible, bean scooping | Grand entrance, "coffee cathedral" |
| 2F | Princi Bakery & Café | Italian bakery, pizza, full food menu | Open kitchen, fresh bread/pizza | Casual dining, chef observation |
| 3F | Experiential Coffee Bar | Exploratory brewing (siphon, Chemex, pour-over) | Coffee flights, origin flights | Laboratory feel, educational |
| 4F | Arriviamo Cocktail Bar | Coffee cocktails, spirits, beer, wine | Espresso Martini, coffee-infused bourbon | Evening destination, bar atmosphere |
| 5F | Rooftop Terrace | Seasonal outdoor seating | Michigan Ave views | Open air, summer-focused |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FLOOR STRUCTURE DATA
# ══════════════════════════════════════════════════════════════════════

floors = pd.DataFrame([
    {'floor': 1, 'name': 'Main Reserve Bar & Scooping Bar',
     'function': 'Coffee, pastries, retail',
     'key_feature': 'Roasting operations, bean scooping, Chicago-exclusive merchandise',
     'atmosphere': 'Grand entrance, coffee cathedral',
     'has_food': True, 'has_coffee': True, 'has_alcohol': False, 'has_retail': True,
     'peak_time': 'morning', 'estimated_dwell_min': 15,
     'description': 'Primary coffee bar visible from street. Reserve-exclusive drinks, artisanal pastries, freshly roasted beans for purchase.'},
    
    {'floor': 2, 'name': 'Princi Bakery & Café',
     'function': 'Italian bakery, full food menu',
     'key_feature': 'Open kitchen, pizza, fresh bread, breakfast/lunch',
     'atmosphere': 'Casual dining, chef observation',
     'has_food': True, 'has_coffee': True, 'has_alcohol': False, 'has_retail': False,
     'peak_time': 'lunch', 'estimated_dwell_min': 25,
     'description': 'Princi Italian bakery with open kitchen. Full breakfast/lunch menu including focaccia, salads, pastries. Extended food options vs 1F.'},
    
    {'floor': 3, 'name': 'Experiential Coffee Bar',
     'function': 'Exploratory brewing methods',
     'key_feature': 'Siphon, Chemex, pour-over, coffee flights, origin flights',
     'atmosphere': 'Laboratory, educational',
     'has_food': True, 'has_coffee': True, 'has_alcohol': False, 'has_retail': False,
     'peak_time': 'afternoon', 'estimated_dwell_min': 20,
     'description': 'Coffee laboratory showcasing rare brewing methods. Siphon flights, Chemex for two, origin tasting flights. Educational experience deepening coffee knowledge.'},
    
    {'floor': 4, 'name': 'Arriviamo Cocktail Bar',
     'function': 'Coffee cocktails, spirits, beer, wine',
     'key_feature': 'Espresso Martini, coffee-infused bourbon, cocktail flights',
     'atmosphere': 'Evening destination, bar',
     'has_food': True, 'has_coffee': True, 'has_alcohol': True, 'has_retail': False,
     'peak_time': 'evening', 'estimated_dwell_min': 35,
     'description': 'Craft cocktail bar with coffee-infused spirits. Espresso Martini flights, Roastery Old Fashioned, local Chicago mixologist creations. Full food menu available.'},
    
    {'floor': 5, 'name': 'Rooftop Terrace',
     'function': 'Seasonal outdoor seating',
     'key_feature': 'Michigan Avenue views, open air',
     'atmosphere': 'Open air, seasonal',
     'has_food': False, 'has_coffee': True, 'has_alcohol': True, 'has_retail': False,
     'peak_time': 'afternoon', 'estimated_dwell_min': 20,
     'description': 'Seasonal rooftop with Michigan Ave views. Coffee and cocktail service. Open primarily in summer months (May-September).'},
])

floors.to_csv(OUT_DIR / 'roastery_floors.csv', index=False)
print(f'Saved: roastery_floors.csv ({len(floors)} floors)')
display(floors[['floor', 'name', 'function', 'peak_time', 'estimated_dwell_min']])

## Section 2 — Menu Items by Floor

Menu data sourced from [Starbucks Reserve official menus](https://www.starbucksreserve.com/menus). Prices are estimated from public sources (media reviews, visitor reports) since the official site loads prices dynamically.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MENU DATA BY FLOOR
# ══════════════════════════════════════════════════════════════════════

menu_items = [
    # === 1F: Main Reserve Bar ===
    {'floor': 1, 'item': 'Brewed Coffee', 'category': 'coffee', 'cal_min': 10, 'cal_max': 10, 'price_est': 4.50, 'is_signature': False},
    {'floor': 1, 'item': 'Cold Brew', 'category': 'coffee', 'cal_min': 120, 'cal_max': 120, 'price_est': 5.50, 'is_signature': False},
    {'floor': 1, 'item': 'Nitro Cold Brew', 'category': 'coffee', 'cal_min': 120, 'cal_max': 120, 'price_est': 6.00, 'is_signature': False},
    {'floor': 1, 'item': 'Latte', 'category': 'coffee', 'cal_min': 110, 'cal_max': 230, 'price_est': 6.00, 'is_signature': False},
    {'floor': 1, 'item': 'Cappuccino', 'category': 'coffee', 'cal_min': 100, 'cal_max': 210, 'price_est': 5.75, 'is_signature': False},
    {'floor': 1, 'item': 'Tiramisu Latte', 'category': 'coffee_creation', 'cal_min': 380, 'cal_max': 600, 'price_est': 7.50, 'is_signature': True},
    {'floor': 1, 'item': 'Iced Ube Coconut Latte', 'category': 'coffee_creation', 'cal_min': 220, 'cal_max': 250, 'price_est': 7.50, 'is_signature': True},
    {'floor': 1, 'item': 'Iced Blooming Yuzu Espresso', 'category': 'coffee_creation', 'cal_min': 190, 'cal_max': 250, 'price_est': 7.50, 'is_signature': True},
    {'floor': 1, 'item': 'Salted Caramel Bianco Latte', 'category': 'coffee_creation', 'cal_min': 130, 'cal_max': 390, 'price_est': 7.00, 'is_signature': True},
    {'floor': 1, 'item': 'Whiskey Barrel-Aged Cold Brew (Spirit-Free)', 'category': 'coffee', 'cal_min': 30, 'cal_max': 30, 'price_est': 7.00, 'is_signature': True},
    {'floor': 1, 'item': 'Matcha Latte', 'category': 'tea', 'cal_min': 160, 'cal_max': 300, 'price_est': 6.50, 'is_signature': False},
    {'floor': 1, 'item': 'Masala Chai Latte', 'category': 'tea', 'cal_min': 110, 'cal_max': 250, 'price_est': 6.00, 'is_signature': False},
    {'floor': 1, 'item': 'Croissant', 'category': 'pastry', 'cal_min': 250, 'cal_max': 350, 'price_est': 4.50, 'is_signature': False},
    {'floor': 1, 'item': 'Tiramisu', 'category': 'pastry', 'cal_min': 300, 'cal_max': 400, 'price_est': 6.50, 'is_signature': False},
    {'floor': 1, 'item': 'Whiskey Barrel-Aged Doughnut', 'category': 'pastry', 'cal_min': 350, 'cal_max': 450, 'price_est': 5.50, 'is_signature': True},
    # Beans
    {'floor': 1, 'item': 'Honduras CAFICO (beans)', 'category': 'retail', 'cal_min': 0, 'cal_max': 0, 'price_est': 25.00, 'is_signature': True},
    {'floor': 1, 'item': 'Chicago Roastery Microblend (beans)', 'category': 'retail', 'cal_min': 0, 'cal_max': 0, 'price_est': 28.00, 'is_signature': True},
    {'floor': 1, 'item': 'Whiskey Barrel-Aged Guatemala (beans)', 'category': 'retail', 'cal_min': 0, 'cal_max': 0, 'price_est': 35.00, 'is_signature': True},
    
    # === 2F: Princi Bakery ===
    {'floor': 2, 'item': 'Margherita Focaccia', 'category': 'food', 'cal_min': 400, 'cal_max': 500, 'price_est': 8.50, 'is_signature': False},
    {'floor': 2, 'item': 'Pepperoni Focaccia', 'category': 'food', 'cal_min': 450, 'cal_max': 550, 'price_est': 9.00, 'is_signature': False},
    {'floor': 2, 'item': 'Prosciutto & Bufala Focaccia', 'category': 'food', 'cal_min': 400, 'cal_max': 500, 'price_est': 10.00, 'is_signature': True},
    {'floor': 2, 'item': 'Kale Caesar Salad', 'category': 'food', 'cal_min': 250, 'cal_max': 350, 'price_est': 9.50, 'is_signature': False},
    {'floor': 2, 'item': 'Porchetta & Egg On Ciabatta', 'category': 'food', 'cal_min': 450, 'cal_max': 550, 'price_est': 9.00, 'is_signature': True},
    {'floor': 2, 'item': 'Avocado Toast on Seeded Sourdough', 'category': 'food', 'cal_min': 300, 'cal_max': 400, 'price_est': 8.00, 'is_signature': False},
    {'floor': 2, 'item': 'Salmon & Cream Cheese Toast', 'category': 'food', 'cal_min': 350, 'cal_max': 450, 'price_est': 9.50, 'is_signature': False},
    {'floor': 2, 'item': 'Eggs In Purgatory', 'category': 'food', 'cal_min': 300, 'cal_max': 400, 'price_est': 8.50, 'is_signature': True},
    {'floor': 2, 'item': 'Granola, Yogurt & Fruit', 'category': 'food', 'cal_min': 200, 'cal_max': 300, 'price_est': 7.00, 'is_signature': False},
    {'floor': 2, 'item': 'Herbed Parm Twists', 'category': 'food', 'cal_min': 200, 'cal_max': 300, 'price_est': 5.50, 'is_signature': False},
    
    # === 3F: Experiential Coffee Bar ===
    {'floor': 3, 'item': 'Siphon Flight', 'category': 'experience', 'cal_min': 10, 'cal_max': 15, 'price_est': 15.00, 'is_signature': True},
    {'floor': 3, 'item': 'Origin Flight', 'category': 'experience', 'cal_min': 15, 'cal_max': 15, 'price_est': 12.00, 'is_signature': True},
    {'floor': 3, 'item': 'Chemex Brewed Coffee For Two', 'category': 'experience', 'cal_min': 15, 'cal_max': 15, 'price_est': 12.00, 'is_signature': True},
    {'floor': 3, 'item': 'Pour-Over', 'category': 'experience', 'cal_min': 10, 'cal_max': 10, 'price_est': 8.00, 'is_signature': True},
    {'floor': 3, 'item': 'Cold Brew Malt', 'category': 'coffee_creation', 'cal_min': 450, 'cal_max': 450, 'price_est': 9.00, 'is_signature': True},
    {'floor': 3, 'item': 'Classic Affogato', 'category': 'coffee_creation', 'cal_min': 230, 'cal_max': 230, 'price_est': 8.00, 'is_signature': True},
    {'floor': 3, 'item': 'Caramel Mocha Drizzle Affogato', 'category': 'coffee_creation', 'cal_min': 300, 'cal_max': 300, 'price_est': 9.00, 'is_signature': True},
    
    # === 4F: Arriviamo Bar ===
    {'floor': 4, 'item': 'Espresso Martini', 'category': 'cocktail', 'cal_min': 200, 'cal_max': 250, 'price_est': 16.00, 'is_signature': True},
    {'floor': 4, 'item': 'Espresso Martini Flight', 'category': 'cocktail', 'cal_min': 400, 'cal_max': 500, 'price_est': 22.00, 'is_signature': True},
    {'floor': 4, 'item': 'Ube Espresso Martini', 'category': 'cocktail', 'cal_min': 250, 'cal_max': 300, 'price_est': 17.00, 'is_signature': True},
    {'floor': 4, 'item': 'Roastery Old Fashioned', 'category': 'cocktail', 'cal_min': 180, 'cal_max': 220, 'price_est': 18.00, 'is_signature': True},
    {'floor': 4, 'item': 'Starbucks Reserve Boulevardier', 'category': 'cocktail', 'cal_min': 200, 'cal_max': 250, 'price_est': 17.00, 'is_signature': True},
    {'floor': 4, 'item': 'Double Matcha Margarita', 'category': 'cocktail', 'cal_min': 180, 'cal_max': 220, 'price_est': 16.00, 'is_signature': True},
    {'floor': 4, 'item': 'French 75', 'category': 'cocktail', 'cal_min': 150, 'cal_max': 180, 'price_est': 16.00, 'is_signature': False},
    {'floor': 4, 'item': 'Manhattan', 'category': 'cocktail', 'cal_min': 180, 'cal_max': 200, 'price_est': 16.00, 'is_signature': False},
    {'floor': 4, 'item': 'Aperol Spritz', 'category': 'cocktail', 'cal_min': 150, 'cal_max': 180, 'price_est': 14.00, 'is_signature': False},
    {'floor': 4, 'item': 'Seasonal Cocktail Flight', 'category': 'cocktail', 'cal_min': 500, 'cal_max': 600, 'price_est': 24.00, 'is_signature': True},
    {'floor': 4, 'item': 'Whiskey Barrel-Aged Cold Brew Comparison Flight', 'category': 'cocktail', 'cal_min': 60, 'cal_max': 100, 'price_est': 15.00, 'is_signature': True},
    {'floor': 4, 'item': 'Beer (Pilsner/IPA/Seasonal)', 'category': 'alcohol', 'cal_min': 150, 'cal_max': 250, 'price_est': 8.00, 'is_signature': False},
    {'floor': 4, 'item': 'Wine (Red/White/Rosé)', 'category': 'alcohol', 'cal_min': 120, 'cal_max': 180, 'price_est': 12.00, 'is_signature': False},
    
    # === 5F: Rooftop ===
    {'floor': 5, 'item': 'Seasonal Coffee Beverage', 'category': 'coffee', 'cal_min': 100, 'cal_max': 300, 'price_est': 7.00, 'is_signature': False},
    {'floor': 5, 'item': 'Seasonal Cocktail', 'category': 'cocktail', 'cal_min': 180, 'cal_max': 250, 'price_est': 16.00, 'is_signature': False},
]

menu_df = pd.DataFrame(menu_items)
menu_df.to_csv(OUT_DIR / 'roastery_menu.csv', index=False)
print(f'Saved: roastery_menu.csv ({len(menu_df)} items)')
print(f'\nItems per floor:')
print(menu_df.groupby('floor')['item'].count())
print(f'\nCategories: {menu_df.category.unique().tolist()}')

## Section 3 — Floor Adjacency & Connectivity

The building has a central staircase and elevators connecting all floors. For the ABM, we model floor-to-floor transitions as a weighted adjacency matrix where adjacent floors have higher transition probability.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FLOOR ADJACENCY MATRIX
# ══════════════════════════════════════════════════════════════════════

# Physical adjacency: adjacent floors are easier to reach
# This is the STRUCTURAL prior — will be updated with review-derived behavioral data
floor_labels = ['1F', '2F', '3F', '4F', '5F']

# Base adjacency (physical distance penalty)
# Adjacent floors: high connectivity
# Skip-one: moderate
# Skip-two+: low
adjacency = np.array([
    # 1F    2F    3F    4F    5F
    [0.00, 0.40, 0.15, 0.05, 0.02],  # From 1F
    [0.30, 0.00, 0.35, 0.10, 0.03],  # From 2F
    [0.15, 0.25, 0.00, 0.30, 0.08],  # From 3F
    [0.05, 0.10, 0.25, 0.00, 0.25],  # From 4F
    [0.03, 0.05, 0.10, 0.35, 0.00],  # From 5F
])

# Add "exit" probability (remaining probability)
exit_probs = 1.0 - adjacency.sum(axis=1)
print('Exit probability per floor:')
for i, label in enumerate(floor_labels):
    print(f'  {label}: {exit_probs[i]:.2f}')

# Save adjacency matrix
adj_df = pd.DataFrame(adjacency, index=floor_labels, columns=floor_labels)
adj_df['exit'] = exit_probs
adj_df.to_csv(OUT_DIR / 'floor_adjacency_prior.csv')
print(f'\nSaved: floor_adjacency_prior.csv')

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(adjacency, cmap='Greens', vmin=0, vmax=0.5)

ax.set_xticks(range(5))
ax.set_yticks(range(5))
ax.set_xticklabels(floor_labels)
ax.set_yticklabels(floor_labels)
ax.set_xlabel('To Floor')
ax.set_ylabel('From Floor')
ax.set_title('Floor Transition Prior (Physical Adjacency)', fontsize=13, fontweight='bold')

for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{adjacency[i,j]:.2f}', ha='center', va='center', fontsize=10,
                color='white' if adjacency[i,j] > 0.25 else 'black')

plt.colorbar(im, label='Transition Probability')
plt.tight_layout()
plt.show()

## Section 4 — Visualizations

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FLOOR & MENU VISUALIZATIONS
# ══════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Items per floor
floor_counts = menu_df.groupby('floor')['item'].count()
colors_floor = ['#00704A', '#1E3932', '#D4E9E2', '#CBA258', '#A0522D']
axes[0, 0].barh([f'{i}F' for i in floor_counts.index], floor_counts.values, color=colors_floor)
axes[0, 0].set_title('Menu Items per Floor')
axes[0, 0].set_xlabel('Number of Items')
for i, v in enumerate(floor_counts.values):
    axes[0, 0].text(v + 0.3, i, str(v), va='center', fontweight='bold')

# 2. Category distribution
cat_counts = menu_df['category'].value_counts()
axes[0, 1].barh(cat_counts.index, cat_counts.values, color='#00704A', edgecolor='white')
axes[0, 1].set_title('Items by Category')
axes[0, 1].set_xlabel('Count')

# 3. Price range by floor
floor_prices = menu_df.groupby('floor')['price_est'].agg(['mean', 'min', 'max'])
axes[1, 0].bar([f'{i}F' for i in floor_prices.index], floor_prices['mean'], 
               color=colors_floor, edgecolor='white', alpha=0.8)
axes[1, 0].errorbar([f'{i}F' for i in floor_prices.index], floor_prices['mean'],
                    yerr=[floor_prices['mean'] - floor_prices['min'], 
                          floor_prices['max'] - floor_prices['mean']],
                    fmt='none', color='black', capsize=5)
axes[1, 0].set_title('Price Range by Floor ($)')
axes[1, 0].set_ylabel('Price ($)')

# 4. Estimated dwell time
dwell = floors[['floor', 'name', 'estimated_dwell_min']]
axes[1, 1].barh([f'{f}F: {n}' for f, n in zip(dwell['floor'], dwell['name'])],
                dwell['estimated_dwell_min'], color=colors_floor)
axes[1, 1].set_title('Estimated Dwell Time per Floor')
axes[1, 1].set_xlabel('Minutes')
for i, v in enumerate(dwell['estimated_dwell_min']):
    axes[1, 1].text(v + 0.5, i, f'{v} min', va='center', fontsize=9)

plt.suptitle('Chicago Roastery: Structure & Menu Profile', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary stats
print('\n=== Summary ===')
print(f'Total floors: {len(floors)}')
print(f'Total menu items: {len(menu_df)}')
print(f'Signature items: {menu_df.is_signature.sum()} ({menu_df.is_signature.mean():.0%})')
print(f'Price range: ${menu_df.price_est.min():.2f} - ${menu_df.price_est.max():.2f}')
print(f'Mean price: ${menu_df.price_est.mean():.2f}')
print(f'Total estimated visit time: {floors.estimated_dwell_min.sum()} min (all floors)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SCHEMATIC FLOOR MAP
# ══════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 12))

floor_data = [
    (5, 'Rooftop Terrace', '#87CEEB', 'Seasonal\nViews'),
    (4, 'Arriviamo Bar', '#CBA258', 'Cocktails\nEspresso Martini'),
    (3, 'Experiential Coffee', '#D4E9E2', 'Siphon\nFlights'),
    (2, 'Princi Bakery', '#F4A460', 'Pizza\nPastries'),
    (1, 'Main Reserve Bar', '#00704A', 'Entry\nRoasting'),
]

for i, (floor, name, color, features) in enumerate(floor_data):
    y = i * 2.2
    rect = mpatches.FancyBboxPatch((0.5, y), 7, 1.8, boxstyle='round,pad=0.1',
                                     facecolor=color, edgecolor='black', linewidth=2, alpha=0.8)
    ax.add_patch(rect)
    ax.text(1.0, y + 0.9, f'{floor}F', fontsize=14, fontweight='bold', va='center',
            color='white' if color in ['#00704A', '#CBA258'] else 'black')
    ax.text(3.0, y + 1.2, name, fontsize=11, fontweight='bold', va='center',
            color='white' if color in ['#00704A', '#CBA258'] else 'black')
    ax.text(3.0, y + 0.5, features, fontsize=9, va='center',
            color='white' if color in ['#00704A', '#CBA258'] else '#333')
    
    # Arrows between floors
    if i > 0:
        ax.annotate('', xy=(4, y), xytext=(4, y - 0.4),
                    arrowprops=dict(arrowstyle='<->', color='#333', lw=1.5))

ax.set_xlim(0, 8)
ax.set_ylim(-0.5, 11.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Starbucks Reserve Roastery Chicago\n35,000 sq ft | 646 N Michigan Ave',
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## Data Files Created

| File | Rows | Description |
|------|------|-------------|
| `roastery_floors.csv` | 5 | Floor structure, features, dwell times |
| `roastery_menu.csv` | 46 | Menu items by floor with nutrition and estimated prices |
| `floor_adjacency_prior.csv` | 5×6 | Physical adjacency-based transition matrix (prior) |

## Next Steps

- **Notebook 2:** NLP analysis of Yelp/Google reviews to extract floor mentions, product mentions, and sentiment
- **Notebook 3:** Update transition matrix with review-derived behavioral data
- **Notebook 4:** Agent-Based Model simulation

## Data Sources

- [Starbucks Reserve Official Site](https://www.starbucksreserve.com/locations/chicago-roastery)
- [TravelFamilyBlog Floor Guide](https://travelfamilyblog.com/starbucks-reserve-roastery-chicago-a-complete-guide-to-every-floor/)
- [Wikipedia: Starbucks Reserve Roastery Chicago](https://en.wikipedia.org/wiki/Starbucks_Reserve_Roastery_(Chicago))
- [Choose Chicago](https://www.choosechicago.com/blog/worlds-largest-starbucks-chicago/)